Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

Load Dataset

In [ ]:
events = pd.read_csv("events.csv")
item_properties = pd.read_csv("item_properties.csv")
category_tree = pd.read_csv("category_tree.csv")



print("Events Data Info:")
print(events.info())
print("\nSample Events Data:")
print(events.head())

In [ ]:
# Remove duplicate rows
events.drop_duplicates(inplace=True)

# Remove rows with missing user or item IDs
events.dropna(subset=['visitorid', 'itemid'], inplace=True)

# Convert timestamp to datetime format
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')


# Remove users with very few interactions (noise reduction)

user_interaction_count = events['visitorid'].value_counts()
active_users = user_interaction_count[user_interaction_count > 5].index
events = events[events['visitorid'].isin(active_users)]


# Assign weights to interaction types

interaction_weights = {
    'view': 1,
    'addtocart': 2,
    'transaction': 3
}

events['interaction_score'] = events['event'].map(interaction_weights)

# Remove rows with unknown interaction types (if any)
events.dropna(subset=['interaction_score'], inplace=True)


# Aggregate interactions per user-item pair

interaction_df = events.groupby(
    ['visitorid', 'itemid']
)['interaction_score'].sum().reset_index()

print("\nUser-Item Interaction Table:")
print(interaction_df.head())


# Build User–Item Interaction Matrix


user_item_matrix = interaction_df.pivot_table(
    index='visitorid',
    columns='itemid',
    values='interaction_score',
    fill_value=0
)

print("\nUser-Item Interaction Matrix Shape:")
print(user_item_matrix.shape)


# Encode User and Item IDs

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

interaction_df['user_id'] = user_encoder.fit_transform(interaction_df['visitorid'])
interaction_df['item_id'] = item_encoder.fit_transform(interaction_df['itemid'])

# Save Cleaned & Processed Data


events.to_csv("cleaned_events.csv", index=False)
interaction_df.to_csv("user_item_interactions.csv", index=False)
user_item_matrix.to_csv("user_item_matrix.csv")


Milestone 2

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Step 1: Prepare data for Surprise
data = events[['visitorid', 'itemid', 'rating']]
reader = Reader(rating_scale=(1,3))

# Step 2: Train-test split
trainset, testset = train_test_split(Dataset.load_from_df(data, reader), test_size=0.2)

# Step 3: Train SVD model
algo = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)
algo.fit(trainset)
print("\nSVD Model training completed!")

# Step 4: Test model
predictions = algo.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)
print(f"Test RMSE: {rmse:.3f}, Test MAE: {mae:.3f}")

# Step 5: Generate Top-N recommendations for each user
def get_top_n(predictions, n=5):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

top_n = get_top_n(predictions, n=5)
sample_user = list(top_n.keys())[0]
print(f"\nTop 5 recommended items for user {sample_user}:")
print(top_n[sample_user])


Milestone 3

In [ ]:
# Step 1: Create hold-out test data (last interaction per user)
train_data = events.copy()
test_data = defaultdict(list)
for user, group in events.groupby('visitorid'):
    last_event = group.iloc[-1]
    test_data[user].append(last_event['itemid'])
    train_data.drop(last_event.name, inplace=True)

# Step 2: Define Precision, Recall, F1
def precision_recall_f1(recommended_items, relevant_items):
    recommended_items = set(recommended_items)
    relevant_items = set(relevant_items)
    tp = len(recommended_items & relevant_items)
    precision = tp / len(recommended_items) if recommended_items else 0
    recall = tp / len(relevant_items) if relevant_items else 0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) else 0
    return precision, recall, f1

# Step 3: Evaluate Top-N recommendations
precisions, recalls, f1s = [], [], []

for user in test_data:
    if user not in top_n:
        continue
    recs = [iid for (iid, _) in top_n[user]]
    precision, recall, f1 = precision_recall_f1(recs, test_data[user])
    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)

print("\nTop-N Model Performance:")
print(f"Average Precision: {np.mean(precisions):.3f}")
print(f"Average Recall: {np.mean(recalls):.3f}")
print(f"Average F1-score: {np.mean(f1s):.3f}")